# Strands 에이전트와 AgentCore 메모리 (도구를 통한 장기 메모리)

## 소개

이 노트북은 AgentCore 메모리와 Strands 프레임워크를 사용하여 **공유 장기 메모리를 갖춘 멀티 에이전트 시스템**을 구현하는 방법을 보여줍니다. 여러 전문 에이전트가 각 에이전트 전용 네임스페이스가 있는 공통 장기 메모리 저장소에 접근하면서 함께 작동하는 방법을 살펴봅니다.

### 튜토리얼 상세 정보

| 항목                | 세부 사항                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 장기 대화형                                                                      |
| 에이전트 유형       | 여행 예약 어시스턴트                                                              |
| 에이전틱 프레임워크 | Strands Agents                                                                   |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소  | AgentCore 사용자 선호도 메모리 추출, 메모리 저장 및 조회용 도구                    |
| 예제 난이도         | 중급                                                                             |

학습 내용:

- 장기 메모리 전략을 갖춘 공유 메모리 리소스 설정 방법
- 자체 메모리 네임스페이스에 접근할 수 있는 전문 에이전트 생성
- 전문 에이전트에 위임하는 조정(코디네이터) 에이전트 구현
- 에이전트 전문화를 위한 구조화된 메모리 네임스페이스 활용

## 시나리오 컨텍스트

이 예제에서는 다음으로 구성된 **여행 계획 시스템**을 만듭니다:
1. 여행 선호도 및 이력에 대한 장기 메모리를 가진 항공편 예약 어시스턴트
2. 숙소 선호도에 대한 장기 메모리를 가진 호텔 예약 어시스턴트
3. 이러한 전문 에이전트를 조율하는 여행 코디네이터

각 전문 에이전트는 공통 메모리 저장소 내에서 자체 네임스페이스에 접근하여, 시간이 지남에 따라 사용자 선호도에 대한 지속적인 이해를 구축할 수 있습니다. 이 접근 방식은 복잡한 도메인을 메모리 인프라를 공유하면서도 각자의 전문 영역을 유지하는 전문 에이전트로 분할하는 방법을 보여줍니다.

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구사항

이 튜토리얼을 실행하려면 다음이 필요합니다:
- Python 3.10+
- Amazon Bedrock AgentCore 메모리 권한이 있는 AWS 자격 증명
- Amazon Bedrock AgentCore SDK

환경을 설정하고 공유 장기 메모리 리소스를 만드는 것부터 시작하겠습니다!

## 1단계: 환경 설정
필요한 모든 라이브러리를 가져오고 이 노트북을 실행하는 데 필요한 클라이언트를 정의하는 것부터 시작하겠습니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
import time
from datetime import datetime
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("travel-assistant")

Amazon Bedrock 모델과 AgentCore에 대한 적절한 권한이 있는 리전을 정의합니다.

In [ ]:
region = "us-west-2" # AWS 리전으로 교체하세요
MODEL_ID= "global.anthropic.claude-haiku-4-5-20251001-v1:0"

## 2단계: 공유 메모리 리소스 생성
이 섹션에서는 각 에이전트 전용 네임스페이스가 있는 공통 장기 메모리 저장소를 생성합니다.

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

client = MemoryClient(region_name=region)
memory_name = "TravelAgent_LTM"
memory_id = None

In [ ]:
from botocore.exceptions import ClientError

try:
    print("장기 메모리 전략으로 메모리 생성 중...")
    # 단일 장기 메모리 전략으로 메모리 리소스 생성
    # {actorId} 플레이스홀더는 실제 액터 ID로 동적 대체됩니다
    memory = client.create_memory_and_wait(
        name=memory_name,
        description="Travel Agent with Long-Term Memory",
        strategies=[{
            StrategyType.USER_PREFERENCE.value: {
                "name": "UserPreferences",
                "description": "Captures user preferences",
                "namespaces": ["travel/{actorId}/preferences/"]
            }
        }],
        event_expiry_days=7,  # 단기 대화는 7일 후 만료
        max_wait=300,
        poll_interval=10
    )

    # 메모리 ID 추출 및 출력
    memory_id = memory['id']
    print(f"메모리가 성공적으로 생성되었습니다. ID: {memory_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # 메모리가 이미 존재하면 해당 ID를 조회
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"메모리가 이미 존재합니다. 기존 메모리 ID 사용: {memory_id}")
except Exception as e:
    # 메모리 생성 중 오류 처리
    logger.info(f"오류: {e}")
    import traceback
    traceback.print_exc()
    # 오류 시 정리 - 부분적으로 생성된 메모리 삭제
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"메모리 정리 완료: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"메모리 정리 실패: {cleanup_error}")

### 장기 메모리 전략 이해하기

우리가 생성하는 메모리 리소스는 단일 사용자 선호도 전략을 갖춘 AgentCore 메모리의 장기 메모리 기능을 사용합니다:

1. **사용자 선호도 메모리 전략**: 대화에서 언급된 사용자 선호도를 자동으로 추출하고 통합합니다
2. **액터 기반 네임스페이스**: 네임스페이스 경로에 액터 ID를 사용하여 각 에이전트별 별도의 공간을 생성합니다
3. **메모리 영속성**: 만료되는 단기 메모리와 달리, 추출된 선호도는 대화 만료 이후에도 유지됩니다

네임스페이스 패턴 `travel/{actorId}/preferences/`를 사용하면, 각 전문 에이전트는 액터 ID를 기반으로 고유한 네임스페이스를 갖게 됩니다:
- 항공편 에이전트가 접근하는 경로: `travel/flight-user-TIMESTAMP/preferences/`
- 호텔 에이전트가 접근하는 경로: `travel/hotel-user-TIMESTAMP/preferences/`

이를 통해 각 에이전트가 공통 메모리 인프라를 사용하면서도 자체적인 전문 지식을 유지할 수 있습니다.

### 에이전트 아이덴티티 설정

In [ ]:
# 각 전문 에이전트에 고유한 액터 ID를 생성하되 세션 ID는 공유
flight_actor_id = f"flight-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
hotel_actor_id = f"hotel-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"
flight_namespace = f"travel/{flight_actor_id}/preferences/"
hotel_namespace = f"travel/{hotel_actor_id}/preferences/"

In [ ]:
# 필요한 컴포넌트 가져오기
from strands import Agent, tool
from strands_tools.agent_core_memory import AgentCoreMemoryToolProvider

### 3단계: 메모리 훅 프로바이더 생성

이 단계에서는 메모리 작업을 자동화하는 커스텀 `MemoryHookProvider` 클래스를 정의합니다. 훅은 에이전트 실행 생명주기의 특정 시점에서 실행되는 특수 함수입니다. 우리가 생성하는 메모리 훅은 하나의 주요 기능을 수행합니다:

1. **메모리 저장**: 에이전트가 응답한 후 새로운 대화를 저장합니다

이를 통해 수동 관리 없이 원활한 메모리 경험을 만들 수 있습니다.

In [ ]:
class MemoryHookProvider(HookProvider):
    """자동 메모리 관리를 위한 훅 프로바이더"""
    
    def __init__(self, memory_id: str, client: MemoryClient):
        self.memory_id = memory_id
        self.client = client
    
    def save_memories(self, event: AfterInvocationEvent):
        """에이전트 응답 후 대화 저장"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2:
                # 마지막 사용자 및 어시스턴트 메시지 가져오기
                user_msg = None
                assistant_msg = None
                
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not assistant_msg:
                        assistant_msg = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not user_msg and "toolResult" not in msg["content"][0]:
                        user_msg = msg["content"][0]["text"]
                        break
                
                if user_msg and assistant_msg:
                    # 에이전트 상태에서 세션 정보 가져오기
                    actor_id = event.agent.state.get("actor_id")
                    session_id = event.agent.state.get("session_id")
                    
                    if not actor_id or not session_id:
                        logger.warning("에이전트 상태에 actor_id 또는 session_id가 없습니다")
                        return
                    
                    # 대화 저장
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=actor_id,
                        session_id=session_id,
                        messages=[(user_msg, "USER"), (assistant_msg, "ASSISTANT")]
                    )
                    logger.info("대화를 메모리에 저장했습니다")
                    
        except Exception as e:
            logger.error(f"메모리 저장 실패: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        """메모리 훅 등록"""
        registry.add_callback(AfterInvocationEvent, self.save_memories)
        logger.info("메모리 훅이 등록되었습니다")

### 전문 에이전트에 메모리 접근 권한 부여

다음으로, 전문 에이전트를 위한 시스템 프롬프트를 정의합니다.

In [ ]:
# 시스템 프롬프트: 호텔 예약 전문 에이전트 - 호텔 찾기, 예약, 숙소/편의시설 안내
HOTEL_BOOKING_PROMPT = f"""You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities. 
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner.Ask max two questions per turn. Keep the messages short, don't overwhelm the customer."""

# 시스템 프롬프트: 항공편 예약 전문 에이전트 - 항공편 찾기, 예약, 항공사/노선/여행 정책 안내
FLIGHT_BOOKING_PROMPT = f"""You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies. 
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner.Ask max two questions per turn. Keep the messages short, don't overwhelm the customer."""

### 에이전트 도구 구현
이제 코디네이터 에이전트가 사용할 수 있는 전문 에이전트를 도구로 구현합니다:

In [ ]:
@tool
def flight_booking_assistant(query: str) -> str:
    """
    Process and respond to flight booking queries.

    Args:
        query: A flight-related question about bookings, schedules, airlines, or travel policies

    Returns:
        Detailed flight information, booking options, or travel advice
    """
    try:
        provider_flight = AgentCoreMemoryToolProvider(
            memory_id=memory_id,      # 필수
            actor_id=flight_actor_id, # 필수
            session_id=session_id,    # 필수
            region = region,
            namespace=flight_namespace
        )
        
        flight_memory_hooks = MemoryHookProvider(memory_id, client)

        flight_agent = Agent(
            tools=provider_flight.tools,
            hooks=[flight_memory_hooks],
            model=MODEL_ID,
            system_prompt=FLIGHT_BOOKING_PROMPT,
            state={"actor_id": flight_actor_id, "session_id": session_id}
        )

        # 에이전트를 호출하고 응답 반환
        response = flight_agent(query)
        return str(response)
    except Exception as e:
        return f"항공편 예약 어시스턴트 오류: {str(e)}"

@tool
def hotel_booking_assistant(query: str) -> str:
    """
    Process and respond to hotel booking queries.

    Args:
        query: A hotel-related question about accommodations, amenities, or reservations

    Returns:
        Detailed hotel information, booking options, or accommodation advice
    """
    try:
        provider_hotel = AgentCoreMemoryToolProvider(
            memory_id=memory_id,      
            actor_id=hotel_actor_id, 
            session_id=session_id,   
            region = region,
            namespace=hotel_namespace
        )

        hotel_memory_hooks = MemoryHookProvider(memory_id, client)

        hotel_booking_agent = Agent(
            tools=provider_hotel.tools,
            hooks=[hotel_memory_hooks],
            model=MODEL_ID,
            system_prompt=HOTEL_BOOKING_PROMPT,
            state={"actor_id": hotel_actor_id, "session_id": session_id}
        )
        
        # 에이전트를 호출하고 응답 반환
        response = hotel_booking_agent(query)
        return str(response)
    except Exception as e:
        return f"호텔 예약 어시스턴트 오류: {str(e)}"

### 코디네이터 에이전트 생성

마지막으로, 이러한 전문 도구들을 조율하는 메인 여행 계획 에이전트를 생성합니다:

In [ ]:
# 시스템 프롬프트: 코디네이터 에이전트 - 항공편/호텔 전문 도구를 조율하는 종합 여행 계획 어시스턴트
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_assistant tool
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_assistant tool
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources. \
Ask max two questions per turn. Keep the messages short, don't overwhelm the customer.
"""

In [ ]:
travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant]
)

## 멀티 에이전트 메모리 시스템 테스트

여행 계획 시나리오로 멀티 에이전트 시스템을 테스트해 보겠습니다:

### 선택 사항: 항공편 예약 어시스턴트의 장기 메모리 초기 데이터 주입
항공편 예약 에이전트에 초기 데이터를 주입하려면 아래 셀의 주석을 해제하세요.

In [ ]:
# 이전 대화 시드 데이터 (메모리에 저장되어 에이전트가 조회하므로 영문 유지)
"""flight_previous_messages = [
    ("Hi, I'm Sarah", "USER"),
    ("Hello Sarah! Welcome to FlightAssist. How can I help you with your travel plans today?", "ASSISTANT"),
    ("I'm looking to book a flight from New York to London sometime next month.", "USER"),
    ("I'd be happy to help you find flights from New York to London for next month. Do you have specific dates in mind, or are you flexible?", "ASSISTANT"),
    ("I'm thinking around the 15th to the 25th, but I can be a bit flexible.", "USER"),
    ("Great! That gives us some room to find the best options. Do you have any preferences regarding airlines or flight times?", "ASSISTANT"),
    ("I definitely prefer direct flights if possible. I really don't like layovers.", "USER"),
    ("I completely understand your preference for direct flights. Layovers can be inconvenient. Fortunately, there are several airlines offering direct flights between New York and London, including British Airways, American Airlines, Delta, and Virgin Atlantic.", "ASSISTANT"),
    ("That's good to hear. I've had good experiences with British Airways in the past.", "USER"),
    ("British Airways does offer excellent service on transatlantic routes. I'll keep that in mind when searching for options. Do you have any seating preferences or other requirements for your flight?", "ASSISTANT"),
    ("I always try to get an aisle seat. I like being able to get up without disturbing others, especially on long flights.", "USER"),
    ("An aisle seat is a great choice for long-haul flights like New York to London. I'll note your preference for aisle seating. Would you prefer to fly in the morning, afternoon, or evening?", "ASSISTANT"),
    ("I prefer overnight flights for long journeys. It helps me adjust to the time difference better.", "USER"),
    ("Overnight flights are indeed a smart choice for eastbound transatlantic travel. They allow you to arrive in London in the morning and help minimize jet lag. British Airways, Delta, and American all offer evening departures from New York that arrive in London the next morning.", "ASSISTANT"),
    ("Perfect! And I'm also wondering about baggage allowances since I'll be staying for about a week.", "USER"),
    ("For a week-long trip, most travelers find that a standard checked bag plus a carry-on is sufficient. British Airways typically allows one free checked bag on transatlantic flights in economy class, plus a carry-on and personal item. Would you like me to check the specific allowances for your preferred dates?", "ASSISTANT")
]

print("\n이전 대화로 메모리를 초기화하는 중...")

# 대화 이력을 단기 메모리에 저장
initial = client.create_event(
    memory_id=memory_id,
    actor_id=flight_actor_id,
    session_id=session_id,
    messages=flight_previous_messages,
)
print("대화가 단기 메모리에 저장되었습니다")"""

In [ ]:
# LA에서 마드리드로의 여행 예약 요청 (7월 1일 ~ 8월 2일)
travel_agent("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")

In [ ]:
# 이베리아 항공 직항 선호 표현
travel_agent("I prefer direct flights with Iberia")

In [ ]:
# 오전 이코노미석 항공편 선호
travel_agent("I would like a flight in the morning, in economy")

In [ ]:
# SNA 공항에서 출발하고 15일 후에 귀국 요청
travel_agent("I would like to fly from SNA, and return 15 days later")

## 메모리 영속성 테스트

메모리 시스템이 올바르게 작동하는지 테스트하기 위해, 새로운 여행 에이전트 인스턴스를 생성하고 이전에 저장된 정보에 접근할 수 있는지 확인합니다:

In [ ]:
time.sleep(60) # 메모리가 이벤트를 처리할 시간을 줍니다..
# 여행 에이전트의 새 인스턴스 생성
new_travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant]
)

# 이전 대화에 대해 질문 (항공편 선호도 확인 요청)
new_travel_agent("Can you remind me about flight preferences?")

## 요약

이 노트북에서는 다음을 시연했습니다:

1. 여러 에이전트를 위한 공유 메모리 리소스 생성 방법
2. 메모리 접근 권한이 있는 전문 에이전트를 도구로 구현하는 방법
3. 대화 컨텍스트를 유지하면서 여러 에이전트 간 조율하는 방법
4. 다른 에이전트 인스턴스 간에도 메모리가 유지되는 방법

공유 메모리를 갖춘 이 멀티 에이전트 아키텍처는 전문 도메인을 처리하면서도 일관된 사용자 경험을 유지하는 복잡한 대화형 AI 시스템을 구축하기 위한 강력한 접근 방식을 제공합니다.

## 정리
이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제하겠습니다.

In [ ]:
# client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
#)